## Data Prep Tests

In [ ]:
import sys
from pathlib import Path

# Add parent dir so we can import the data prep scripts
CODE_DIR = Path("../").resolve()
sys.path.insert(0, str(CODE_DIR))

DATA_DIR    = Path(r"C:\Users\elric\Documents\01 DSBA\05 Term 2\01 AI in Healthcare\02 Final Project\02 Data")
MEDDEC_DIR  = DATA_DIR / "meddec-mimic-iii"
NOTES_PATH  = DATA_DIR / "NOTEEVENTS.csv" / "NOTEEVENTS.csv"
PHENO_PATH  = DATA_DIR / "pheno_annoations_ACTdb102003.csv" 

json_files = list((MEDDEC_DIR / "data").glob("*.json"))
print(f"Data dir:    {DATA_DIR}")
print(f"Count of MedDec JSONs: {len(json_files)} files")

Data dir:    C:\Users\elric\Documents\01 DSBA\05 Term 2\01 AI in Healthcare\02 Final Project\02 Data
Count of MedDec JSONs: 403 files


In [10]:
json_files[:5]

[WindowsPath('C:/Users/elric/Documents/01 DSBA/05 Term 2/01 AI in Healthcare/02 Final Project/02 Data/meddec-mimic-iii/data/10814_101543_52781.json'),
 WindowsPath('C:/Users/elric/Documents/01 DSBA/05 Term 2/01 AI in Healthcare/02 Final Project/02 Data/meddec-mimic-iii/data/10814_119849_52793.json'),
 WindowsPath('C:/Users/elric/Documents/01 DSBA/05 Term 2/01 AI in Healthcare/02 Final Project/02 Data/meddec-mimic-iii/data/10814_141014_52776.json'),
 WindowsPath('C:/Users/elric/Documents/01 DSBA/05 Term 2/01 AI in Healthcare/02 Final Project/02 Data/meddec-mimic-iii/data/10814_171527_52779.json'),
 WindowsPath('C:/Users/elric/Documents/01 DSBA/05 Term 2/01 AI in Healthcare/02 Final Project/02 Data/meddec-mimic-iii/data/10814_180749_52780.json')]

### Extract Raw Texts

In [5]:
from extract_texts import extract_texts

n_extracted, n_missing = extract_texts(MEDDEC_DIR, NOTES_PATH)

Extracted 403 notes to C:\Users\elric\Documents\01 DSBA\05 Term 2\01 AI in Healthcare\02 Final Project\02 Data\meddec-mimic-iii\raw_text  (0 missing)


In [ ]:
import json

raw_text_dir = MEDDEC_DIR / "raw_text"

# Spot-check one file offsets from annotation should match the raw text
# a. annotation
sample_json = json_files[0]
data = json.loads(sample_json.read_text())
ann = data["annotations"][0]
start, end = int(ann["start_offset"]), int(ann["end_offset"])
# b. raw text
sample_txt = raw_text_dir / f"{sample_json.stem}.txt"
note_text = sample_txt.read_text(encoding="utf-8")
extracted_span = note_text[start:end]

print(f"Annotation decision : '{ann['decision']}'")
print(f"Text at offsets     : '{extracted_span}'")
assert ann["decision"].lower() in extracted_span.lower() or extracted_span.lower() in ann["decision"].lower(), (
    f"Offset mismatch! decision='{ann['decision']}' but text[{start}:{end}]='{extracted_span}'"
)
print("PASS: Character offsets align with annotation text.")

Annotation decision : 'Bacteremia'
Text at offsets     : 'Bacteremia'
PASS: Character offsets align with annotation text.


### Build the stats csv

In [13]:
from build_stats import build_stats

stats = build_stats(DATA_DIR)
print(stats)

Saved 403 rows to C:\Users\elric\Documents\01 DSBA\05 Term 2\01 AI in Healthcare\02 Final Project\02 Data\meddec-mimic-iii\stats.csv
     SUBJECT_ID  HADM_ID GENDER              ETHNICITY LANGUAGE
0         10997   105782      M                  WHITE      NaN
1         11559   103284      M                  WHITE     ENGL
2         11398   109233      F                  WHITE      NaN
3         11894   123607      M  UNKNOWN/NOT SPECIFIED      NaN
4         12310   185464      M                  WHITE     RUSS
..          ...      ...    ...                    ...      ...
398       95722   150858      F                  WHITE     ENGL
399       99809   154672      M                  WHITE     ENGL
400       99830   176834      M  UNKNOWN/NOT SPECIFIED     SPAN
401       93799   182669      M          BLACK/AFRICAN     ENGL
402       96234   167807      M                  WHITE     ENGL

[403 rows x 5 columns]


In [ ]:
expected_cols = {"SUBJECT_ID", "HADM_ID", "GENDER", "ETHNICITY", "LANGUAGE"}
assert expected_cols == set(stats.columns), f"Column mismatch: {stats.columns.tolist()}" # column match
assert len(stats) > 0, "stats.csv is empty!" # stats.csv is populated

# All (SUBJECT_ID, HADM_ID) pairs in stats should come from MedDec
meddec_pairs = {tuple(map(int, f.stem.split("_")[:2])) for f in json_files}
stats_pairs  = set(zip(stats["SUBJECT_ID"], stats["HADM_ID"]))
extra = stats_pairs - meddec_pairs
assert not extra, f"stats.csv contains pairs not in MedDec: {list(extra)[:5]}"

print(f"PASS: {len(stats)} rows, correct columns, all IDs from MedDec.")
print(stats[["GENDER", "ETHNICITY", "LANGUAGE"]].describe(include="all").T[["count", "unique", "top"]])

PASS: 403 rows, correct columns, all IDs from MedDec.
          count unique    top
GENDER      403      2      M
ETHNICITY   403     17  WHITE
LANGUAGE    271     11   ENGL


### Phenotype preprocessing

In [25]:
from preprocess_phenos import preprocess_phenos

In [ ]:
# Save phenos.csv
phenos = preprocess_phenos(PHENO_PATH, output_file=MEDDEC_DIR / "phenos.csv")
print(phenos)

Saved 844 rows to C:\Users\elric\Documents\01 DSBA\05 Term 2\01 AI in Healthcare\02 Final Project\02 Data\meddec-mimic-iii\phenos.csv
     SUBJECT_ID  HADM_ID  ROW_ID  \
0           109   164029   15322   
1           154   102354    7847   
2           154   162891    7846   
3           188   160697   20255   
4           188   191517   20256   
..          ...      ...     ...   
839       81444   100960    7834   
840       85999   134661   36826   
841       88206   155516   51338   
842       94525   154715   54366   
843       95722   150858   34672   

                                       phenotype_label OPERATOR  
0                                                 NONE      JTW  
1                     DEPRESSION,PSYCHIATRIC.DISORDERS      JTW  
2                                                 NONE      JTW  
3                                                    ?   JF,JTW  
4                                                 NONE      JTW  
..                                   

In [ ]:
assert "phenotype_label" in phenos.columns, "Missing 'phenotype_label' column"
assert len(phenos) > 0, "phenos.csv is empty!"
assert phenos["phenotype_label"].notna().all(), "Some rows have NaN phenotype_label"

print("Label distribution:")
print(phenos["phenotype_label"].value_counts().head(10))

# Verify UNSURE handling: only flag notes where the PRIORITY annotator marked UNSURE.
# (Lower-priority annotator UNSURE is correctly ignored by the aggregation logic.)
import pandas as pd

OPERATOR_TIER = {"DAG": 0, "PAT": 0, "JTW": 1, "JF": 1, "ETM": 2, "JW": 2}

raw = pd.read_csv(PHENO_PATH)
raw["PSYCHIATRIC.DISORDERS"] = (
    raw["DEMENTIA"] | raw["DEVELOPMENTAL.DELAY.RETARDATION"] | raw["SCHIZOPHRENIA.AND.OTHER.PSYCHIATRIC.DISORDERS"]
)
raw["_tier"] = raw["OPERATOR"].map(OPERATOR_TIER)

def priority_has_unsure(group):
    best_tier = group["_tier"].min()
    return group[group["_tier"] == best_tier]["UNSURE"].sum() > 0

priority_unsure = (
    raw.groupby(["SUBJECT_ID", "HADM_ID", "ROW_ID"])
    .apply(priority_has_unsure)
    .reset_index(name="priority_unsure")
)
should_be_uncertain = priority_unsure[priority_unsure["priority_unsure"]]
merged = should_be_uncertain.merge(phenos, on=["SUBJECT_ID", "HADM_ID", "ROW_ID"])
wrong  = merged[~merged["phenotype_label"].str.contains(r"\?", na=False)]

assert len(wrong) == 0, f"{len(wrong)} notes where priority annotator marked UNSURE but label is not '?':\n{wrong.head()}"
print(f"PASS: {len(merged)} priority-UNSURE notes all map to '?'.")
print(f"      {len(priority_unsure) - len(should_be_uncertain)} notes had UNSURE only from lower-priority annotators — correctly ignored.")
print(f"PASS: {len(phenos)} unique notes processed.")

### Build Train/Val/Test Splits

In [18]:
from build_splits import build_splits

train, val, test = build_splits(MEDDEC_DIR)

Train: 324 files (301 subjects)
Val:   38 files (37 subjects)
Test:  41 files (39 subjects)


In [20]:
print(train[:5])

['10814_101543_52781.json', '10814_119849_52793.json', '10814_141014_52776.json', '10814_171527_52779.json', '10814_180749_52780.json']


In [21]:
# Check: Split file count = total file count)
total = len(train) + len(val) + len(test)
assert total == len(json_files), f"Split total {total} != {len(json_files)} JSON files"

# Check: No file overlap between splits
assert not set(train) & set(val),  "Overlap between train and val!"
assert not set(train) & set(test), "Overlap between train and test!"
assert not set(val)   & set(test), "Overlap between val and test!"

# Check: No subject-level leakage: subjects in train must not appear in val/test
def subjects_of(names):
    return {n.split("_")[0] for n in names}

train_subs = subjects_of(train)
val_subs   = subjects_of(val)
test_subs  = subjects_of(test)
assert not train_subs & val_subs,  "Subject leakage: train/val share subjects!"
assert not train_subs & test_subs, "Subject leakage: train/test share subjects!"
assert not val_subs   & test_subs, "Subject leakage: val/test share subjects!"

# Check split fractions
n = len(json_files)
print(f"Train: {len(train):3d}/{n}  ({100*len(train)/n:.1f}%)")
print(f"Val:   {len(val):3d}/{n}  ({100*len(val)/n:.1f}%)")
print(f"Test:  {len(test):3d}/{n}  ({100*len(test)/n:.1f}%)")
print("PASS: No overlaps, no subject leakage.")

Train: 324/403  (80.4%)
Val:    38/403  (9.4%)
Test:   41/403  (10.2%)
PASS: No overlaps, no subject leakage.


---
## Phase 2 — ELECTRA Token Classification

### 2.0 — Environment Setup

**If running on Colab**, uncomment and run the cell below to mount Drive and point paths at your uploaded data.  
**If running locally**, skip it — `CODE_DIR` and `DATA_DIR` from the top of this notebook are already correct.

In [ ]:
# ── Colab only: mount Drive and override paths ───────────────────────────────
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR   = Path("/content/drive/MyDrive/02 Data")
# MEDDEC_DIR = DATA_DIR / "meddec-mimic-iii"
# CODE_DIR   = Path("/content/drive/MyDrive/04 Code")
# sys.path.insert(0, str(CODE_DIR))
# !pip install -q torch transformers

print("CODE_DIR :", CODE_DIR)
print("DATA_DIR :", DATA_DIR)

---
### 2.1 — Dataset (`dataset.py`)

#### Part A — Logic checks (no GPU / no torch needed)

These tests verify the three pure-Python helpers in isolation using a mock tokenizer,
so you can run them locally even before installing `torch`.

In [ ]:
from dataset import parse_category, char_to_token_safe, build_label_sequence, LABEL_O, LABEL_PAD

# ── parse_category ────────────────────────────────────────────────────────────
assert parse_category("Category 1: Contact-related") == 0
assert parse_category("Category 5: Drug")            == 4
assert parse_category("Category 9: Advice")          == 8
assert parse_category("Category 10: Legal")          is None   # excluded
assert parse_category("TBD")                         is None
print("PASS: parse_category")

In [ ]:
# ── char_to_token_safe + build_label_sequence  (mock tokenizer) ───────────────
#
# Simulated note: "hello world"  (11 chars, 0-indexed)
#   chars  0-4  = "hello" → token 1
#   char   5    = " "     → None  (whitespace, not a token)
#   chars  6-10 = "world" → tokens 2-3  (subword split: "wor", "ld")
#   token  0    = [CLS]   → token_to_chars returns None
#   token  4    = [SEP]   → token_to_chars returns None

char_tok = {0:1, 1:1, 2:1, 3:1, 4:1,   # "hello" → token 1
            5:None,                       # space   → no token
            6:2, 7:2, 8:2,               # "wor"   → token 2
            9:3, 10:3}                    # "ld"    → token 3

class MockEncoding:
    def char_to_token(self, pos):      return char_tok.get(pos)
    def token_to_chars(self, tok):     return None if tok in (0, 4) else object()

enc = MockEncoding()

# Forward nudge: space at char 5 should resolve to token 2 ("w")
assert char_to_token_safe(enc, 5, nudge_forward=True)  == 2
# Backward nudge: space at char 5 should resolve to token 1 ("o" of "hello")
assert char_to_token_safe(enc, 5, nudge_forward=False) == 1
print("PASS: char_to_token_safe")

# Annotation: "hello" = chars 0..5 (exclusive), Category 5: Drug (0-indexed cat = 4)
# Expected: token 1 → B=8 (4*2), tokens 2-3 → O (no annotation covers them), 0/4 → PAD
annotations = [{"start_offset": "0", "end_offset": "5", "category": "Category 5: Drug"}]
labels = build_label_sequence(enc, annotations, seq_len=5)
print("Labels:", labels)
#   token 0 (CLS) → -100,  token 1 → 8 (B-Drug),  tokens 2-3 → 18 (O),  token 4 (SEP) → -100
assert labels == [-100, 8, 18, 18, -100], f"Unexpected: {labels}"
print("PASS: build_label_sequence")

#### Part B — Full dataset tests (requires `torch` + `transformers`)

Install once if needed:
```
pip install torch transformers
```
Then restart the kernel and re-run from the top.

In [ ]:
from dataset import load_electra_tokenizer

# Downloads ~440 MB on first run, cached locally afterwards
tokenizer = load_electra_tokenizer()
print("Vocabulary size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

# Quick sanity check: subword tokenisation of a clinical phrase
sample = "start insulin therapy"
tokens = tokenizer.tokenize(sample)
print(f"\n'{sample}' → {tokens}")

In [ ]:
from dataset import MedDecDataset, NUM_LABELS, LABEL_O, LABEL_PAD
from collections import Counter

SPLITS_DIR = MEDDEC_DIR / "splits"

train_ds = MedDecDataset(SPLITS_DIR / "train.txt", MEDDEC_DIR, tokenizer, train=True)
val_ds   = MedDecDataset(SPLITS_DIR / "val.txt",   MEDDEC_DIR, tokenizer, train=False)
test_ds  = MedDecDataset(SPLITS_DIR / "test.txt",  MEDDEC_DIR, tokenizer, train=False)

print(f"Train: {len(train_ds)} notes")
print(f"Val:   {len(val_ds)} notes")
print(f"Test:  {len(test_ds)} notes")

# Each item should have the three expected tensor keys
item = train_ds[0]
assert set(item.keys()) >= {"input_ids", "attention_mask", "labels"}
print(f"\nSample item (train, after windowing):")
print(f"  input_ids shape:      {item['input_ids'].shape}    (should be ≤ 512)")
print(f"  attention_mask shape: {item['attention_mask'].shape}")
print(f"  labels shape:         {item['labels'].shape}")
assert item["input_ids"].shape[0] <= 512, "Training window must be ≤ 512 tokens"

# Eval item: full note, may be > 512
item_eval = val_ds[0]
print(f"\nSample item (eval, full note): {item_eval['input_ids'].shape[0]} tokens")
print("PASS: Dataset shapes correct")

In [ ]:
# ── Spot-check: verify a real annotation maps to the correct B label ──────────
#
# Strategy: take the first annotation from the first JSON file, find which token
# it should land on using the tokenizer directly, and confirm the label array
# agrees.

import json

sample_fname = train_ds.samples[0]["file_name"]
sample_stem  = Path(sample_fname).stem
ann_data     = json.loads((MEDDEC_DIR / "data" / sample_fname).read_text())
note_text    = (MEDDEC_DIR / "raw_text" / f"{sample_stem}.txt").read_text(encoding="utf-8")

ann        = ann_data["annotations"][0]
start_char = int(ann["start_offset"])
end_char   = int(ann["end_offset"])
cat        = parse_category(ann["category"])

# Re-tokenise to look up the expected token index
encoding   = tokenizer(note_text, add_special_tokens=True, truncation=False)
enc_start  = char_to_token_safe(encoding, start_char, nudge_forward=True)

# The pre-built label array (stored in the dataset before any windowing)
full_labels = train_ds.samples[0]["labels"]

print(f"Annotation : '{ann['decision']}'")
print(f"Category   : {ann['category']}  →  0-indexed cat = {cat}")
print(f"Chars      : [{start_char}:{end_char}]  text = '{note_text[start_char:end_char]}'")
print(f"Token      : enc_start = {enc_start}")
print(f"Token text : '{tokenizer.decode(encoding['input_ids'][enc_start:enc_start+3]).strip()}'")
print(f"Label stored : {full_labels[enc_start]}  (expected B = {cat * 2})")

assert full_labels[enc_start] == cat * 2, (
    f"B-label mismatch: stored {full_labels[enc_start]}, expected {cat * 2}"
)
print("PASS: Annotation correctly encoded as B label")

In [ ]:
# ── Label distribution across the full training set ──────────────────────────
#
# What we expect to see:
#   - LABEL_O (18) should be by far the most common — most tokens are not decisions
#   - LABEL_PAD (-100) accounts for CLS + SEP tokens (2 per note)
#   - B and I labels should be rare but present for all 9 categories

all_labels = [lbl for s in train_ds.samples for lbl in s["labels"]]
cnt = Counter(all_labels)

CATEGORY_NAMES = [
    "Contact", "Gathering info", "Defining problem", "Treatment goal",
    "Drug", "Therapeutic proc.", "Eval test result", "Deferment", "Advice"
]

total = len(all_labels)
print(f"Total tokens (train, full notes): {total:,}")
print(f"  O (outside)  : {cnt[LABEL_O]:>8,}  ({100*cnt[LABEL_O]/total:.1f}%)")
print(f"  PAD (-100)   : {cnt[LABEL_PAD]:>8,}  ({100*cnt[LABEL_PAD]/total:.1f}%)")
print()
print(f"  {'Cat':>3}  {'B-label':>7}  {'I-label':>7}  {'B count':>8}  {'I count':>8}  Name")
for n in range(9):
    b, i = n * 2, n * 2 + 1
    print(f"  {n+1:>3}  {b:>7}  {i:>7}  {cnt[b]:>8,}  {cnt[i]:>8,}  {CATEGORY_NAMES[n]}")

missing = [n+1 for n in range(9) if cnt[n*2] == 0]
if missing:
    print(f"\nWARNING: categories {missing} have no B-labels in train — check annotations")
else:
    print("\nPASS: All 9 categories have at least one B-label in training data")